# Auto Loader & Structured Streaming - Complete Guide

---

## PART 1: AUTO LOADER

---

### 1.1 What is Auto Loader?

Auto Loader (`cloudFiles`) is a Databricks-optimized Structured Streaming source that **incrementally and efficiently ingests new data files** as they arrive in cloud object storage. It automatically tracks which files have been processed using checkpointing.

---

### 1.2 Compatible Sources (Input)

| Cloud Storage | Protocol | Supported |
| --- | --- | --- |
| Azure Data Lake Storage Gen2 | `abfss://` | YES |
| Azure Blob Storage | `wasbs://` (legacy, deprecated) | YES |
| Amazon S3 | `s3://` | YES |
| Google Cloud Storage | `gs://` | YES |
| Unity Catalog Volumes | `/Volumes/catalog/schema/volume/` | YES |

### Supported File Formats

| Format | Notes |
| --- | --- |
| JSON | Single-line and multi-line |
| CSV | With/without headers |
| PARQUET | Columnar format |
| AVRO | Schema-embedded |
| ORC | Columnar format |
| XML | Requires DBR 14.3+ |
| TEXT | One row per line |
| BINARYFILE | Raw binary content (images, PDFs, etc.) |
| FILE (Beta) | Ingest as FILE references for documents/images |

---

### 1.3 Compatible Sinks (Output)

| Sink | Supported | Notes |
| --- | --- | --- |
| Delta Lake table | YES (primary) | Most common target |
| Apache Iceberg table | YES | Via UniForm |
| Parquet / JSON / CSV files | YES | File-based sinks |
| Kafka | YES | Via writeStream format("kafka") |
| Console (debug) | YES | format("console") |
| Memory (debug) | YES | format("memory") |
| foreachBatch | YES | Custom sink logic |
| forEach | YES | Row-by-row custom logic |

---

### 1.4 File Discovery Modes

| Mode | Setup Complexity | Scalability | Cost | When to Use |
| --- | --- | --- | --- | --- |
| **File Events** (recommended) | Low (one-time permission) | Millions of files/hour | Lowest | Default for most workloads |
| **Classic File Notification** | High (21+ cloud configs) | Millions of files/hour | Medium | When file events unavailable |
| **Directory Listing** | None | Limited by directory size | Highest (LIST API costs) | Small dirs, one-time backfills |

---

### 1.5 Auto Loader Options - MANDATORY

| Option | Description | Example |
| --- | --- | --- |
| `cloudFiles.format` | **REQUIRED** - File format to read | `"json"`, `"csv"`, `"parquet"`, `"avro"` |
| `path` (in `.load()`) | **REQUIRED** - Source directory path | `"abfss://container@storage.dfs.core.windows.net/data/"` |
| `checkpointLocation` (in writeStream) | **REQUIRED** - Checkpoint directory for tracking progress | `"/checkpoints/my_stream"` |

### Schema Options (at least one required for schema-less formats)

| Option | Description |
| --- | --- |
| `cloudFiles.schemaLocation` | Directory to store inferred schema (enables schema inference & evolution) |
| `.schema(schema)` | Provide explicit schema (no inference) |

---

### 1.6 Auto Loader Options - OPTIONAL (Common)

| Option | Default | Description |
| --- | --- | --- |
| `cloudFiles.schemaEvolutionMode` | `addNewColumns` | How to handle schema changes: `addNewColumns`, `rescue`, `failOnNewColumns`, `none` |
| `cloudFiles.inferColumnTypes` | `false` | Infer actual types instead of treating all as STRING |
| `cloudFiles.maxFilesPerTrigger` | 1000 | Max new files to process per micro-batch |
| `cloudFiles.maxBytesPerTrigger` | None | Max bytes to process per micro-batch |
| `cloudFiles.schemaHints` | None | Provide type hints for specific columns: `"col1 INT, col2 TIMESTAMP"` |
| `cloudFiles.allowOverwrites` | `false` | Whether to allow file overwrites to be reprocessed |
| `cloudFiles.validateOptions` | `true` | Validate provided options |
| `cloudFiles.rescuedDataColumn` | None | Column name for data that doesn't match schema (e.g., `"_rescued_data"`) |
| `cloudFiles.partitionColumns` | None | Auto-detect partition columns from directory structure |

---

### 1.7 Options Specific to DIRECTORY LISTING Mode

| Option | Default | Description |
| --- | --- | --- |
| `cloudFiles.useIncrementalListing` | `auto` | (Deprecated) Use incremental listing for faster discovery |
| `cloudFiles.includeExistingFiles` | `true` | Process files that already exist when stream starts |
| `cloudFiles.maxFileAge` | None | Ignore files older than this threshold |

---

### 1.8 Options Specific to FILE NOTIFICATION Mode (Classic)

| Option | Default | Description |
| --- | --- | --- |
| `cloudFiles.useNotifications` | `false` | Enable classic file notification mode |
| `cloudFiles.fetchParallelism` | 1 | Threads for fetching messages from queue |
| `cloudFiles.pathRewrites` | None | JSON map for S3 bucket/key path rewrites |
| `cloudFiles.resourceTag.*` | None | Key-value tags for cloud resources |
| `cloudFiles.backfillInterval` | None | Trigger periodic backfills (e.g., `"1 day"`) |

#### Azure-Specific (Classic Notification)
| Option | Description |
| --- | --- |
| `cloudFiles.connectionString` | Azure Storage account connection string |
| `cloudFiles.resourceGroup` | Azure resource group name |
| `cloudFiles.subscriptionId` | Azure subscription ID |
| `cloudFiles.tenantId` | Azure AD tenant ID |
| `cloudFiles.clientId` | Azure AD app client ID |
| `cloudFiles.clientSecret` | Azure AD app client secret |

#### AWS-Specific (Classic Notification)
| Option | Description |
| --- | --- |
| `cloudFiles.region` | AWS region for SNS/SQS |
| `cloudFiles.queueUrl` | Existing SQS queue URL (skip auto-creation) |

---

### 1.9 Options Specific to FILE EVENTS Mode (Managed)

| Option | Default | Description |
| --- | --- | --- |
| `cloudFiles.useManagedFileEvents` | `false` | Enable managed file events mode |

**Unsupported options when using file events:**
* `cloudFiles.useNotifications`
* `cloudFiles.useIncrementalListing`
* `cloudFiles.fetchParallelism`
* `cloudFiles.backfillInterval`
* `cloudFiles.pathRewrites`
* `cloudFiles.resourceTag.*`

---

### 1.10 Options ONLY for Auto Loader (not in standard Structured Streaming)

All `cloudFiles.*` options are exclusive to Auto Loader. They do NOT exist in standard Kafka, Delta, or socket streaming sources:

* `cloudFiles.format`
* `cloudFiles.schemaLocation`
* `cloudFiles.schemaEvolutionMode`
* `cloudFiles.schemaHints`
* `cloudFiles.inferColumnTypes`
* `cloudFiles.maxFilesPerTrigger`
* `cloudFiles.maxBytesPerTrigger`
* `cloudFiles.useNotifications`
* `cloudFiles.useManagedFileEvents`
* `cloudFiles.includeExistingFiles`
* `cloudFiles.allowOverwrites`
* `cloudFiles.rescuedDataColumn`
* `cloudFiles.partitionColumns`
* `cloudFiles.cleanSource`
* `cloudFiles.backfillInterval`

---

### 1.11 Auto Loader Basic Example

```python
# Auto Loader with schema inference
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/checkpoints/my_schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load("/Volumes/catalog/schema/volume/raw_data/")
)

# Write to Delta table
(df.writeStream
    .option("checkpointLocation", "/checkpoints/my_stream")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("catalog.schema.bronze_table")
)
```

---
---

## PART 2: STRUCTURED STREAMING

---

### 2.1 What is Structured Streaming?

A scalable, fault-tolerant stream processing engine built on the Spark SQL engine. Treats a live data stream as a table that is being continuously appended to.

---

### 2.2 Compatible Sources (Streaming Inputs)

| Source | Format String | Key Options |
| --- | --- | --- |
| Auto Loader (Cloud Files) | `cloudFiles` | `cloudFiles.format`, path |
| Delta Lake | `delta` | table name or path |
| Apache Kafka | `kafka` | `kafka.bootstrap.servers`, `subscribe` |
| Azure Event Hubs | `kafka` (Kafka-compatible) | Kafka connection with Event Hubs endpoint |
| Amazon Kinesis | `kinesis` | `streamName`, `region` |
| Apache Pulsar | `pulsar` | `service.url`, `topics` |
| Socket (debug) | `socket` | `host`, `port` |
| Rate (testing) | `rate` | `rowsPerSecond` |
| Rate per micro-batch | `rate-micro-batch` | `rowsPerBatch` |

---

### 2.3 Compatible Sinks (Streaming Outputs)

| Sink | Format String | Output Modes Supported |
| --- | --- | --- |
| Delta Lake | `delta` | Append, Complete, Update |
| Apache Kafka | `kafka` | Append, Update, Complete |
| Parquet/JSON/CSV/ORC files | `parquet`/`json`/`csv`/`orc` | Append only |
| Console (debug) | `console` | All modes |
| Memory (debug) | `memory` | All modes |
| foreachBatch | custom function | All modes |
| forEach | custom function | All modes |
| Iceberg | `iceberg` | Append |
| Noop (benchmarking) | `noop` | All modes |

---

### 2.4 Output Modes

| Mode | Behavior | Use Case |
| --- | --- | --- |
| **Append** (default) | Only NEW rows that will never change are emitted | Stateless ETL, file sinks, after watermark expiry |
| **Update** | Only CHANGED rows since last trigger are emitted | Real-time dashboards, Kafka sinks |
| **Complete** | ENTIRE result table is emitted every trigger | Aggregations to console/memory, small result sets |

---

### 2.5 Trigger Types

| Trigger | Syntax | Behavior |
| --- | --- | --- |
| **Micro-batch (default)** | No trigger specified | Process as soon as previous batch completes |
| **Fixed interval** | `trigger(processingTime="10 seconds")` | Process at fixed intervals |
| **Once** (deprecated) | `trigger(once=True)` | Process all available data then stop |
| **AvailableNow** | `trigger(availableNow=True)` | Process all available data in multiple batches, then stop |
| **Continuous** (experimental) | `trigger(continuous="1 second")` | Low-latency (~1ms) continuous processing |

---

### 2.6 Structured Streaming Options (Common)

#### Read Options
| Option | Applicable To | Description |
| --- | --- | --- |
| `maxFilesPerTrigger` | Delta, File sources | Max files per micro-batch |
| `maxBytesPerTrigger` | Delta | Max bytes per micro-batch |
| `startingOffsets` | Kafka | Where to start: `earliest`, `latest`, or JSON offsets |
| `endingOffsets` | Kafka (batch) | Where to end reading |
| `subscribe` / `subscribePattern` | Kafka | Topics to subscribe |
| `kafka.bootstrap.servers` | Kafka | Kafka broker addresses |
| `ignoreChanges` | Delta | Ignore file changes (updates/deletes) |
| `ignoreDeletes` | Delta | Ignore file deletions |
| `readChangeFeed` | Delta | Read Change Data Feed |
| `startingVersion` / `startingTimestamp` | Delta | Start streaming from specific version/time |

#### Write Options
| Option | Description |
| --- | --- |
| `checkpointLocation` | **REQUIRED** - Fault-tolerance checkpoint directory |
| `mergeSchema` | Allow schema evolution on write (Delta) |
| `queryName` | Name for the streaming query (monitoring) |
| `topic` | Kafka topic to write to |

---
---

## PART 3: STREAM-STATIC JOINS

---

### 3.1 What Is It?
Joining a **streaming DataFrame** with a **static (batch) DataFrame**. The static side is re-read on each trigger.

### 3.2 Supported Join Types

| Join Type | Supported | Notes |
| --- | --- | --- |
| INNER | YES | Stream drives the join |
| LEFT OUTER (stream on left) | YES | Stream is the left table |
| RIGHT OUTER (stream on right) | NO | Would require buffering unbounded stream |
| FULL OUTER | NO | Not supported |
| LEFT SEMI | YES | Filter stream by static existence |
| LEFT ANTI | YES | Filter stream by static non-existence |

### 3.3 Key Characteristics

* **No watermark required** — static data is finite
* **No state management** — only the stream side is incremental
* **Static DataFrame re-read each trigger** — picks up changes to dimension tables
* **Stateless operation** — very efficient, no state store growth

### 3.4 Example
```python
# Static DataFrame (dimension table)
dim_customers = spark.read.table("catalog.schema.dim_customers")

# Streaming DataFrame (fact events)
stream_orders = spark.readStream.table("catalog.schema.orders")

# Stream-Static Join
enriched = stream_orders.join(dim_customers, "customer_id", "left")

(enriched.writeStream
    .option("checkpointLocation", "/checkpoints/enriched_orders")
    .trigger(availableNow=True)
    .toTable("catalog.schema.enriched_orders")
)
```

### 3.5 Limitations
* Static side is read at EACH trigger — if source changes frequently, this can be expensive
* If static side is very large, consider broadcasting or caching
* No exactly-once guarantee on the static side (if it changes mid-batch)

---
---

## PART 4: STREAM-STREAM JOINS

---

### 4.1 What Is It?
Joining **two streaming DataFrames** together. Both sides are unbounded and arrive incrementally.

### 4.2 Supported Join Types

| Join Type | Watermark Required? | Notes |
| --- | --- | --- |
| INNER | Recommended (state cleanup) | Works without watermark but state grows unbounded |
| LEFT OUTER | YES (on right stream) | Emit null when right side not matched within watermark |
| RIGHT OUTER | YES (on left stream) | Emit null when left side not matched within watermark |
| FULL OUTER | YES (both streams) | Emit nulls for both unmatched sides |
| LEFT SEMI | YES (on right stream) | Filter left stream by right stream existence |

### 4.3 How It Works (Internals)

```
┌─────────────────────────────────────────────────────────────┐
│              STREAM-STREAM JOIN ENGINE                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Stream A (Left)         Stream B (Right)                   │
│       │                        │                            │
│       ▼                        ▼                            │
│  ┌─────────┐              ┌─────────┐                       │
│  │ State   │              │ State   │                       │
│  │ Store A │              │ Store B │                       │
│  │ (buffer)│              │ (buffer)│                       │
│  └────┬────┘              └────┬────┘                       │
│       │                        │                            │
│       └──────── JOIN ──────────┘                            │
│                  │                                          │
│                  ▼                                          │
│         ┌───────────────┐                                   │
│         │ Watermark     │                                   │
│         │ State Cleanup │                                   │
│         └───────────────┘                                   │
│                  │                                          │
│                  ▼                                          │
│           Output Results                                    │
└─────────────────────────────────────────────────────────────┘
```

1. **Buffer both sides** in state stores
2. **For each new row on one side**, probe the other side's state for matches
3. **Emit matched rows** immediately (for inner joins)
4. **Watermark advances** → old state beyond watermark boundary is evicted
5. **For outer joins**, unmatched rows are emitted with nulls when watermark guarantees no future match is possible

### 4.4 Watermark & Time Constraints

```python
# BOTH streams must have watermarks for state cleanup
impressions = (spark.readStream.table("impressions")
    .withWatermark("impression_time", "2 hours")
)

clicks = (spark.readStream.table("clicks")
    .withWatermark("click_time", "3 hours")
)

# Join with time range constraint (CRITICAL for bounded state)
joined = impressions.join(
    clicks,
    expr("""
        click_ad_id = impression_ad_id AND
        click_time >= impression_time AND
        click_time <= impression_time + interval 1 hour
    """),
    "leftOuter"
)
```

### 4.5 State Cleanup Rules
* Engine uses **watermark + time range condition** to determine when state can be evicted
* Without time constraints → state grows **unbounded** (OOM risk)
* Watermark = `max(event_time_seen) - threshold`
* State evicted when: `watermark > state_row_event_time + join_time_range`

---
---

## PART 5: MATERIALIZED VIEWS

---

### 5.1 What Is a Materialized View?
A **precomputed result set** that is stored physically (as a Delta table) and refreshed on-demand or on schedule. Unlike a regular view (which re-executes the query every time), a materialized view serves results from the precomputed data.

### 5.2 Materialized View vs Streaming Table

| Aspect | Materialized View | Streaming Table |
| --- | --- | --- |
| Source data | Any (static or streaming) | Append-only streaming sources |
| Refresh logic | Recompute results (full or incremental) | Process new data incrementally |
| Supports UPDATE/DELETE on source | YES | NO (append-only semantics) |
| Output mode equivalent | Complete mode (full result) | Append mode |
| State management | Managed by platform | Checkpoint-based |
| Best for | Aggregations, KPIs, dashboards | ETL, ingestion, pipelines |

### 5.3 How Materialized Views Are Refreshed

#### Refresh Types
| Type | Command | Behavior |
| --- | --- | --- |
| **REFRESH** (default) | `REFRESH MATERIALIZED VIEW mv` | Attempts **incremental refresh** first; falls back to full recompute if needed |
| **FULL REFRESH** | `REFRESH MATERIALIZED VIEW mv FULL` | Always recomputes ALL data from scratch; resets checkpoints |

#### Incremental Refresh (Serverless Only)
* Detects changes in source data
* Recomputes only affected results
* Supported SQL operations: SELECT, WHERE, JOIN, GROUP BY, UNION ALL, window functions
* Requires serverless compute
* NOT supported for all query patterns (e.g., non-deterministic functions)

#### Refresh Scheduling
```sql
-- Create with schedule
CREATE MATERIALIZED VIEW catalog.schema.my_mv
SCHEDULE EVERY 1 HOUR
AS SELECT region, SUM(revenue) as total_revenue
FROM catalog.schema.sales
GROUP BY region;

-- Refresh on source table update
ALTER MATERIALIZED VIEW catalog.schema.my_mv
ADD TRIGGER ON UPDATE;

-- Cron schedule
ALTER MATERIALIZED VIEW catalog.schema.my_mv
ADD SCHEDULE CRON '0 0 * * * ? *';  -- every hour

-- Manual refresh
REFRESH MATERIALIZED VIEW catalog.schema.my_mv;
```

---
---

## PART 6: STATE MANAGEMENT & CHECKPOINTING

---

### 6.1 Checkpoint Directory Structure

```
checkpoint_location/
├── metadata/           # Query ID, configuration
├── offsets/            # Source offsets per micro-batch
│   ├── 0              # Batch 0 offset info
│   ├── 1              # Batch 1 offset info
│   └── ...            
├── commits/           # Which batches are committed (exactly-once)
│   ├── 0
│   ├── 1
│   └── ...
├── state/             # Stateful operator state (if applicable)
│   ├── 0/            # Partition 0
│   │   ├── 1.delta   # State updates for batch 1
│   │   └── 1.snapshot # Full state snapshot
│   └── ...
└── sources/           # Source-specific metadata
    └── 0/            # Source 0 (e.g., Auto Loader file tracking)
```

### 6.2 What Each Component Stores

| Component | Contents | Purpose |
| --- | --- | --- |
| **metadata/** | Query ID (UUID), Spark version | Uniquely identifies the query; prevents accidental reuse |
| **offsets/** | Per-batch source positions (Kafka offsets, file lists, Delta versions) | Know exactly where to resume if interrupted |
| **commits/** | Batch completion markers | Ensures exactly-once: if offset exists but commit doesn't → reprocess batch |
| **state/** | Key-value state for stateful ops (aggregations, joins, dedup) | Maintains running aggregates, buffered join sides, dedup keys |
| **sources/** | Source-specific tracking (Auto Loader: discovered files; Delta: processed versions) | Track what input data has been seen/processed |

### 6.3 State Store Providers

| Provider | Storage | Performance | Best For |
| --- | --- | --- | --- |
| **HDFSBackedStateStore** (default for OSS) | JVM Heap (in-memory) + HDFS snapshots | Fast for small state; GC issues at scale | < 1M keys in state |
| **RocksDB StateStore** (recommended) | Native memory + local SSD + HDFS checkpoints | Handles millions of keys efficiently | Large state, production |

### 6.4 RocksDB State Store (Recommended for Databricks)

```python
# Enable RocksDB state store
spark.conf.set(
    "spark.sql.streaming.stateStore.providerClass",
    "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider"
)

# Enable changelog checkpointing (reduces checkpoint duration)
spark.conf.set(
    "spark.sql.streaming.stateStore.rocksdb.changelogCheckpointing.enabled", 
    "true"
)
```

#### RocksDB Key Configurations
| Config | Default | Description |
| --- | --- | --- |
| `rocksdb.compactOnCommit` | false | Compact RocksDB on each commit |
| `rocksdb.changelogCheckpointing.enabled` | false | Upload changelog instead of full snapshot (faster) |
| `rocksdb.blockSizeKB` | 4 | Block size for RocksDB tables |
| `rocksdb.blockCacheSizeMB` | 8 | Size of block cache |
| `rocksdb.lockAcquireTimeoutMs` | 60000 | Lock timeout for load operations |
| `rocksdb.maxOpenFiles` | -1 | Max open file descriptors (-1 = unlimited) |
| `rocksdb.trackTotalNumberOfRows` | true | Track total rows (disable for performance) |

### 6.5 Changelog Checkpointing vs Snapshot Checkpointing

| Aspect | Snapshot (Traditional) | Changelog |
| --- | --- | --- |
| What's saved | Full state snapshot each batch | Only state changes (deltas) |
| Checkpoint size | Large (full state copy) | Small (only changes) |
| Checkpoint duration | O(total state size) | O(changed keys per batch) |
| Recovery time | Fast (load snapshot directly) | Slightly slower (replay changelogs) |
| Best for | Small state | Large state with few changes per batch |
| Migration | N/A | Backward compatible with snapshots |

---
---

## PART 7: STREAMING INTERNALS

---

### 7.1 Micro-Batch Execution Model

```
┌─────────────────────────────────────────────────────────────────┐
│               MICRO-BATCH EXECUTION LOOP                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌────────────────────────────────────────────────────────────┐ │
│  │ STEP 1: Determine Available Data                          │ │
│  │ - Query sources for new data (offsets/files/records)       │ │
│  │ - Compute start and end offsets for this batch             │ │
│  │ - Write offset range to offsets/ directory (WAL)           │ │
│  └────────────────────────────────────────────────────────────┘ │
│                          │                                      │
│                          ▼                                      │
│  ┌────────────────────────────────────────────────────────────┐ │
│  │ STEP 2: Plan the Batch                                    │ │
│  │ - Create logical plan (SQL/DataFrame query)                │ │
│  │ - Apply Catalyst optimizations                             │ │
│  │ - Generate physical plan (Spark jobs/stages)               │ │
│  └────────────────────────────────────────────────────────────┘ │
│                          │                                      │
│                          ▼                                      │
│  ┌────────────────────────────────────────────────────────────┐ │
│  │ STEP 3: Execute the Batch                                 │ │
│  │ - Read input data within offset range                      │ │
│  │ - Apply transformations (filter, join, aggregate)          │ │
│  │ - Interact with state store (read/update state)            │ │
│  │ - Write results to sink                                    │ │
│  └────────────────────────────────────────────────────────────┘ │
│                          │                                      │
│                          ▼                                      │
│  ┌────────────────────────────────────────────────────────────┐ │
│  │ STEP 4: Commit                                            │ │
│  │ - Checkpoint state store (snapshot or changelog)           │ │
│  │ - Write commit marker to commits/ directory                │ │
│  │ - Advance watermark                                        │ │
│  │ - Report metrics (StreamingQueryProgress)                  │ │
│  └────────────────────────────────────────────────────────────┘ │
│                          │                                      │
│                          ▼                                      │
│              Loop back to STEP 1 (next trigger)                 │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 7.2 Exactly-Once Guarantee (How It Works)

```
Fault Tolerance Flow:

1. OFFSET LOG WRITE (WAL)  →  "I plan to process offsets X to Y"
2. EXECUTE BATCH          →  Process data, update state, write to sink
3. COMMIT LOG WRITE       →  "Batch completed successfully"

Recovery Scenarios:
┌────────────────────────────┬──────────────────────────────────┐
│ Failure Point              │ Recovery Action                  │
├────────────────────────────┼──────────────────────────────────┤
│ Before offset log write    │ Re-determine offsets (no work    │
│                            │ was done)                        │
├────────────────────────────┼──────────────────────────────────┤
│ After offset, before commit│ Re-execute entire batch from     │
│                            │ offset log (idempotent sinks     │
│                            │ prevent duplicates)              │
├────────────────────────────┼──────────────────────────────────┤
│ After commit               │ Move to next batch (this batch   │
│                            │ is complete)                     │
└────────────────────────────┴──────────────────────────────────┘
```

### 7.3 Watermark Internals

```python
# Watermark definition
df.withWatermark("event_time", "10 minutes")
```

**How watermarks work internally:**

| Step | Action |
| --- | --- |
| 1 | Engine tracks `max_event_time_seen` across ALL partitions |
| 2 | Watermark = `max_event_time_seen - threshold` |
| 3 | Watermark only **advances forward** (never goes backward) |
| 4 | Watermark advances at **batch boundaries** (not within a batch) |
| 5 | State older than watermark is eligible for eviction |
| 6 | In Append mode: results are emitted only when watermark passes their window |

**Watermark impact on operations:**
| Operation | Without Watermark | With Watermark |
| --- | --- | --- |
| Windowed Aggregation | State grows forever | State evicted after window + threshold |
| Stream-Stream Join | State grows forever | Old buffered rows evicted |
| Deduplication | Track all keys forever | Only deduplicate within threshold window |
| dropDuplicatesWithinWatermark | N/A | Deduplicate within watermark window |

### 7.4 Stateful Operations Internals

| Operation | State Stored | State Key | State Eviction |
| --- | --- | --- | --- |
| **Streaming Aggregation** | Running aggregate values per group | Group-by keys | When watermark > window end |
| **Stream-Stream Join** | Buffered rows from both sides | Join keys | When watermark > event_time + time_range |
| **Deduplication** | Set of seen keys | Dedup columns | When watermark > event_time |
| **flatMapGroupsWithState** | Custom user-defined state | Group key | User-controlled timeout |
| **transformWithState** (Spark 4.0+) | Custom typed state (value, list, map) | Group key | User-controlled via timers |

### 7.5 State Store Read/Write Flow (Per Micro-Batch)

```
For each stateful operator in the query:

1. LOAD STATE  →  State store loads from last checkpoint
                   (HDFSBacked: deserialize from snapshot)
                   (RocksDB: open local DB or restore from checkpoint)

2. PROCESS DATA → For each input row:
                   - Read current state for the key
                   - Compute new state
                   - Write updated state
                   - Emit output row(s)

3. EVICT STATE → After processing all rows:
                   - Check watermark
                   - Remove state entries older than watermark
                   - For outer joins: emit null-matched rows for evicted state

4. COMMIT STATE → Persist state to durable storage:
                   - Snapshot mode: write full state image
                   - Changelog mode: write only changed keys
                   - Upload to checkpoint location (cloud storage)
```

### 7.6 Streaming Query Lifecycle

```python
# Start a query
query = (df.writeStream
    .format("delta")
    .option("checkpointLocation", "/checkpoints/my_query")
    .outputMode("append")
    .trigger(processingTime="30 seconds")
    .toTable("catalog.schema.output")
)

# Monitor
query.status          # Current status
query.lastProgress    # Last batch metrics
query.recentProgress  # Recent batch metrics array
query.isActive        # Whether query is running

# Stop
query.stop()

# Exception info (if failed)
query.exception
```

### 7.7 Key Metrics (StreamingQueryProgress)

| Metric | Description |
| --- | --- |
| `inputRowsPerSecond` | Rate of data arriving at source |
| `processedRowsPerSecond` | Rate of data being processed |
| `numInputRows` | Total rows in this batch |
| `batchDuration` | Total time for the batch |
| `stateOperators[].numRowsTotal` | Total rows in state store |
| `stateOperators[].numRowsUpdated` | Rows updated in state this batch |
| `stateOperators[].memoryUsedBytes` | Memory used by state |
| `stateOperators[].customMetrics` | Provider-specific metrics (RocksDB) |
| `sources[].startOffset` | Start offset for this batch |
| `sources[].endOffset` | End offset for this batch |
| `sink.numOutputRows` | Rows written to sink |

---
---

## PART 8: ADVANCED CONCEPTS

---

### 8.1 Exactly-Once vs At-Least-Once

| Guarantee | When | Condition |
| --- | --- | --- |
| **Exactly-once** | Idempotent sink + checkpoint | Delta Lake sink (default) |
| **At-least-once** | Non-idempotent sink or reprocessing | Kafka sink (Kafka may duplicate), file sinks on retry |

### 8.2 Handling Late Data

| Strategy | Mechanism |
| --- | --- |
| Accept all late data | No watermark (state grows unbounded) |
| Drop late data beyond threshold | `withWatermark("event_time", "1 hour")` |
| Write late data to separate table | `foreachBatch` with conditional logic |

### 8.3 Schema Evolution in Streaming

| Source | Schema Evolution Support |
| --- | --- |
| Auto Loader | `cloudFiles.schemaEvolutionMode` = `addNewColumns`, `rescue`, `failOnNewColumns`, `none` |
| Delta Lake (source) | Automatic with `schemaTrackingLocation` (DBR 15.2+) |
| Delta Lake (sink) | `.option("mergeSchema", "true")` |
| Kafka | Schema registry integration |

### 8.4 Unsupported Operations in Streaming

| Operation | Reason |
| --- | --- |
| `df.count()` (action) | Cannot return single value from unbounded stream |
| `df.show()` (action) | Use `display()` in Databricks instead |
| `df.sort()` without aggregation | Cannot globally sort infinite stream |
| `LIMIT` / `take(n)` | Undefined on unbounded data |
| Multiple streaming aggregations | Chaining stateful ops not supported (Update/Complete mode) |
| `DISTINCT` | Use `dropDuplicates()` with watermark instead |

### 8.5 Stream-Stream Join vs Stream-Static Join Summary

| Aspect | Stream-Static Join | Stream-Stream Join |
| --- | --- | --- |
| State management | None (stateless) | Both sides buffered in state store |
| Watermark needed | No | Yes (for state cleanup) |
| Late data handling | N/A | Controlled by watermark |
| Static side freshness | Re-read each trigger | N/A (both streaming) |
| Memory growth | Constant | Grows with watermark window |
| Complexity | Low | High |
| Time constraints | Not needed | Required for bounded state |

---
---

## PART 9: PRODUCTION BEST PRACTICES

---

### 9.1 Auto Loader
* Use **file events mode** for production (lowest cost, highest scalability)
* Set `cloudFiles.maxFilesPerTrigger` to control batch sizes
* Enable `cloudFiles.schemaEvolutionMode = "rescue"` for safety
* Use **Unity Catalog Volumes** paths for optimized file discovery
* Set `cloudFiles.inferColumnTypes = true` for non-string types

### 9.2 Structured Streaming
* Always use **RocksDB state store** with changelog checkpointing for stateful queries
* Set **watermarks** on ALL stateful operations
* Use `trigger(availableNow=True)` for cost-effective batch-style streaming
* Monitor `stateOperators[].numRowsTotal` — if growing unbounded, add/tighten watermarks
* Store checkpoints in **reliable cloud storage** (not local/DBFS root)
* Use **separate checkpoints** for each query (never share)
* Enable `spark.sql.streaming.stopTimeout` for graceful shutdowns

### 9.3 State Management
* Enable RocksDB + changelog checkpointing for large state
* Monitor state size via `query.lastProgress`
* Set appropriate watermark thresholds (too large = excess state; too small = data loss)
* Consider `dropDuplicatesWithinWatermark` over `dropDuplicates` for bounded state
* Periodically review state metrics: if `numRowsTotal` is in millions+, verify cleanup is working